# Motor de Recomendacion Musical con Arbol de Busqueda Binaria (BST)
By Cristian Alvear

**Unidad 2 - Tema 2: Arboles Binarios**
las canciones mas **cortas** van a
la **izquierda** y las mas **largas** a la **derecha**.


## Parte 1 - Logica

**Funcionalidad propia (Operacion 4): `canciones_en_rango(min, max)`.** Devuelve las canciones cuya
duracion esta entre un minimo y un maximo (por ejemplo, de 180 a 240 s = temas de 3 a 4 minutos).
Es util para armar listas por tipo de cancion (cortas, medianas, largas) y es eficiente porque usa la
propiedad del BST para no visitar las ramas que no pueden tener resultados.

In [ ]:
# =====================================================================
#  PARTE 1 - LOGICA  (Arbol de Busqueda Binaria con nodos y punteros)
# =====================================================================

class Nodo:
    """Una cancion del arbol: guarda su info y dos punteros (izq y der)."""
    def __init__(self, nombre, duracion):
        self.nombre = nombre        # Nombre de la cancion
        self.duracion = duracion    # Duracion en segundos = CLAVE del BST
        self.izq = None             # Hijo izquierdo (canciones mas cortas)
        self.der = None             # Hijo derecho  (canciones mas largas)


class ArbolMusical:
    """Arbol BST de canciones. Guarda la raiz y todas las operaciones."""

    def __init__(self):
        self.raiz = None            # El arbol empieza vacio

    def agregar(self, nombre, duracion):
        self.raiz = self._insertar(self.raiz, nombre, duracion)

    def _insertar(self, nodo, nombre, duracion):
        if nodo is None:                       # lugar vacio: creamos el nodo
            return Nodo(nombre, duracion)
        if duracion < nodo.duracion:           # mas corta -> izquierda
            nodo.izq = self._insertar(nodo.izq, nombre, duracion)
        else:                                  # mas larga o igual -> derecha
            nodo.der = self._insertar(nodo.der, nombre, duracion)
        return nodo

    # -----------------------------------------------------------------
    #  OPERACION 1: tiempo total de la playlist.
    #  Suma recursiva: nodo actual + subarbol izquierdo + subarbol derecho.
    # -----------------------------------------------------------------
    def calcular_tiempo_total(self, nodo):
        if nodo is None:                       # un lugar vacio aporta 0 s
            return 0
        return (nodo.duracion
                + self.calcular_tiempo_total(nodo.izq)
                + self.calcular_tiempo_total(nodo.der))

    # -----------------------------------------------------------------
    #  OPERACION 2: cancion con duracion exacta o mas cercana.
    #  Baja por UN solo camino del BST (eficiente) guardando la menor
    #  diferencia absoluta encontrada hasta el momento.
    # -----------------------------------------------------------------
    def recomendar_cancion_perfecta(self, nodo, segundos, mejor=None):
        if nodo is None:                       # devolvemos el mejor hallado
            return mejor
        # Si esta cancion se acerca mas que la mejor anterior, la guardamos.
        if mejor is None or abs(nodo.duracion - segundos) < abs(mejor.duracion - segundos):
            mejor = nodo
        if segundos == nodo.duracion:          # coincidencia exacta: ya esta
            return mejor
        if segundos < nodo.duracion:           # bajamos como en una busqueda BST
            return self.recomendar_cancion_perfecta(nodo.izq, segundos, mejor)
        else:
            return self.recomendar_cancion_perfecta(nodo.der, segundos, mejor)

    # -----------------------------------------------------------------
    #  OPERACION 3: eliminar canciones que duren MENOS que 'minimo'.
    #  Truco del BST: si un nodo es demasiado corto, todo su lado
    #  izquierdo tambien lo es (ya quedo vacio), asi que lo reemplazamos
    #  por su subarbol derecho. Los punteros se reconectan solos.
    # -----------------------------------------------------------------
    def filtrar_canciones_cortas(self, minimo):
        self.raiz = self._filtrar(self.raiz, minimo)

    def _filtrar(self, nodo, minimo):
        if nodo is None:
            return None
        nodo.izq = self._filtrar(nodo.izq, minimo)   # filtramos ambos lados
        nodo.der = self._filtrar(nodo.der, minimo)
        if nodo.duracion < minimo:                   # muy corta: la quitamos
            return nodo.der                          # (izq ya quedo vacio)
        return nodo

    # -----------------------------------------------------------------
    #  OPERACION 4: canciones entre min y max.
    #  Usa el BST para PODAR: no entra a la izquierda si el nodo ya no
    #  supera el minimo, ni a la derecha si ya no es menor que el maximo.
    # -----------------------------------------------------------------
    def canciones_en_rango(self, minimo, maximo):
        resultado = []
        self._rango(self.raiz, minimo, maximo, resultado)
        return resultado

    def _rango(self, nodo, minimo, maximo, resultado):
        if nodo is None:
            return
        if nodo.duracion > minimo:                       # puede haber a la izquierda
            self._rango(nodo.izq, minimo, maximo, resultado)
        if minimo <= nodo.duracion <= maximo:            # el nodo entra en el rango
            resultado.append(nodo)
        if nodo.duracion < maximo:                       # puede haber a la derecha
            self._rango(nodo.der, minimo, maximo, resultado)

    # -----------------------------------------------------------------
    #  AUXILIARES: contar canciones y dibujar el arbol como texto
    #  (rama derecha arriba, izquierda abajo; la sangria = profundidad).
    # -----------------------------------------------------------------
    def total_canciones(self, nodo=False):
        if nodo is False:
            nodo = self.raiz
        if nodo is None:
            return 0
        return 1 + self.total_canciones(nodo.izq) + self.total_canciones(nodo.der)

    def dibujar(self, nodo=False, nivel=0):
        if nodo is False:
            nodo = self.raiz
        if nodo is None:
            return ""
        texto  = self.dibujar(nodo.der, nivel + 1)
        texto += "        " * nivel + "[" + str(nodo.duracion) + "s] " + nodo.nombre + "\n"
        texto += self.dibujar(nodo.izq, nivel + 1)
        return texto


# ---------------------- Datos de prueba ------------------------------
arbol = ArbolMusical()
for nombre, dur in [("Blinding Lights", 200), ("Shape of You", 233), ("Imagine", 183),
                    ("Bohemian Rhapsody", 355), ("Intro", 30), ("Stairway to Heaven", 482),
                    ("Billie Jean", 294), ("Get Lucky", 248), ("Interludio corto", 45),
                    ("Nocturne", 210)]:
    arbol.agregar(nombre, dur)

print("Canciones:", arbol.total_canciones())
print("Duracion total:", arbol.calcular_tiempo_total(arbol.raiz), "s\n")
print(arbol.dibujar())


Canciones: 10
Duracion total: 2280 s

                        [482s] Stairway to Heaven
                [355s] Bohemian Rhapsody
                        [294s] Billie Jean
                                [248s] Get Lucky
        [233s] Shape of You
                [210s] Nocturne
[200s] Blinding Lights
        [183s] Imagine
                        [45s] Interludio corto
                [30s] Intro



## Parte 2 - Interfaz (Tkinter)

Sin logica del arbol: cada boton solo lee lo que escribe el usuario, llama a un metodo de
`ArbolMusical` (Parte 1) y muestra el resultado.

In [4]:
# =====================================================================
#  PARTE 2 - INTERFAZ (Tkinter).
# =====================================================================
import tkinter as tk
from tkinter import messagebox

def refrescar():
    """Vuelve a dibujar el arbol y actualiza el resumen de arriba."""
    cuadro.delete("1.0", tk.END)
    cuadro.insert(tk.END, arbol.dibujar())
    info.config(text="Canciones: " + str(arbol.total_canciones())
                     + "   |   Duracion total: " + str(arbol.calcular_tiempo_total(arbol.raiz)) + " s")

def agregar():
    n, d = e_nombre.get().strip(), e_dur.get().strip()
    if n == "" or not d.isdigit():
        return messagebox.showwarning("Datos invalidos", "Escribe un nombre y una duracion entera.")
    arbol.agregar(n, int(d))                                   # Operacion: insertar
    e_nombre.delete(0, tk.END); e_dur.delete(0, tk.END); refrescar()

def tiempo_total():
    t = arbol.calcular_tiempo_total(arbol.raiz)                # Operacion 1
    messagebox.showinfo("Tiempo total", "La playlist dura " + str(t) +
                        " s (" + str(t // 60) + " min " + str(t % 60) + " s).")

def recomendar():
    s = e_rec.get().strip()
    if not s.isdigit():
        return messagebox.showwarning("Dato invalido", "Ingresa un numero entero de segundos.")
    c = arbol.recomendar_cancion_perfecta(arbol.raiz, int(s)) # Operacion 2
    if c is None:
        return messagebox.showinfo("Recomendacion", "El arbol esta vacio.")
    messagebox.showinfo("Cancion recomendada", "'" + c.nombre + "' (" + str(c.duracion) +
                        " s)\nDiferencia: " + str(abs(c.duracion - int(s))) + " s.")

def filtrar():
    s = e_fil.get().strip()
    if not s.isdigit():
        return messagebox.showwarning("Dato invalido", "Ingresa un numero entero de segundos.")
    arbol.filtrar_canciones_cortas(int(s))                    # Operacion 3
    e_fil.delete(0, tk.END); refrescar()
    messagebox.showinfo("Filtro aplicado", "Se eliminaron las canciones de menos de " + s + " s.")

def rango():
    a, b = e_min.get().strip(), e_max.get().strip()
    if not a.isdigit() or not b.isdigit():
        return messagebox.showwarning("Datos invalidos", "Ingresa dos numeros enteros.")
    lista = arbol.canciones_en_rango(int(a), int(b))          # Operacion 4 (propia)
    msg = "\n".join("- " + c.nombre + " (" + str(c.duracion) + " s)" for c in lista) or "Sin resultados."
    messagebox.showinfo("Canciones entre " + a + " y " + b + " s", msg)

# ------------------------------ Ventana ------------------------------
ventana = tk.Tk()
ventana.title("Motor de Recomendacion Musical (BST)")
ventana.configure(bg="#1e1e2e"); ventana.geometry("620x650")

tk.Label(ventana, text="Motor de Recomendacion Musical", font=("Arial", 16, "bold"),
         fg="#ffffff", bg="#1e1e2e").pack(pady=(12, 2))
info = tk.Label(ventana, text="", font=("Arial", 10), fg="#a6e3a1", bg="#1e1e2e"); info.pack()

# Agregar cancion
f1 = tk.LabelFrame(ventana, text="Agregar cancion", fg="#fff", bg="#313244", padx=8, pady=8)
f1.pack(fill="x", padx=12, pady=6)
tk.Label(f1, text="Nombre:", fg="#fff", bg="#313244").grid(row=0, column=0, sticky="e")
e_nombre = tk.Entry(f1, width=26); e_nombre.grid(row=0, column=1, padx=5, pady=3)
tk.Label(f1, text="Duracion (s):", fg="#fff", bg="#313244").grid(row=1, column=0, sticky="e")
e_dur = tk.Entry(f1, width=26); e_dur.grid(row=1, column=1, padx=5, pady=3)
tk.Button(f1, text="Agregar", command=agregar, bg="#89b4fa", fg="#1e1e2e",
          width=12).grid(row=0, column=2, rowspan=2, padx=8)

# Operaciones 1, 2 y 3
f2 = tk.LabelFrame(ventana, text="Operaciones", fg="#fff", bg="#313244", padx=8, pady=8)
f2.pack(fill="x", padx=12, pady=6)
tk.Button(f2, text="Calcular tiempo total (Op. 1)", command=tiempo_total,
          bg="#f9e2af", fg="#1e1e2e").grid(row=0, column=0, columnspan=3, sticky="we", pady=3)
tk.Label(f2, text="Recomendar (s):", fg="#fff", bg="#313244").grid(row=1, column=0, sticky="e", pady=3)
e_rec = tk.Entry(f2, width=10); e_rec.grid(row=1, column=1, sticky="w", padx=5)
tk.Button(f2, text="Buscar (Op. 2)", command=recomendar, bg="#89b4fa", fg="#1e1e2e").grid(row=1, column=2, padx=5)
tk.Label(f2, text="Eliminar menores a (s):", fg="#fff", bg="#313244").grid(row=2, column=0, sticky="e", pady=3)
e_fil = tk.Entry(f2, width=10); e_fil.grid(row=2, column=1, sticky="w", padx=5)
tk.Button(f2, text="Filtrar (Op. 3)", command=filtrar, bg="#f38ba8", fg="#1e1e2e").grid(row=2, column=2, padx=5)

# Operacion 4 (funcionalidad propia)
f3 = tk.LabelFrame(ventana, text="Canciones en rango (Op. 4)", fg="#fff", bg="#313244", padx=8, pady=8)
f3.pack(fill="x", padx=12, pady=6)
tk.Label(f3, text="Desde (s):", fg="#fff", bg="#313244").grid(row=0, column=0, sticky="e")
e_min = tk.Entry(f3, width=8); e_min.grid(row=0, column=1, padx=5)
tk.Label(f3, text="Hasta (s):", fg="#fff", bg="#313244").grid(row=0, column=2, sticky="e")
e_max = tk.Entry(f3, width=8); e_max.grid(row=0, column=3, padx=5)
tk.Button(f3, text="Ver rango", command=rango, bg="#a6e3a1", fg="#1e1e2e").grid(row=0, column=4, padx=8)

# Vista del arbol
f4 = tk.LabelFrame(ventana, text="Estructura del arbol (en memoria)", fg="#fff", bg="#313244", padx=8, pady=8)
f4.pack(fill="both", expand=True, padx=12, pady=6)
sb = tk.Scrollbar(f4); sb.pack(side="right", fill="y")
cuadro = tk.Text(f4, height=12, bg="#11111b", fg="#cdd6f4", font=("Courier New", 10), yscrollcommand=sb.set)
cuadro.pack(fill="both", expand=True); sb.config(command=cuadro.yview)

refrescar()
ventana.mainloop()
